# E02b — blend the main model with XGBoost (validation only)

**Runtime:** a CPU runtime is enough (no GPU needed). Runtime → Run all; about 3–5 minutes.

**Rule fixed before scoring (experiment ledger, E02b):** for each seed s, `p = 0.5 × main model (E01) + 0.5 × XGBoost`,
predicted class = most likely class. Weight 0.5 is not tuned. Deep input = E01, `main_model_v1_news_p41_preds.csv.gz`, already in Drive.

Steps: (1) rerun XGBoost unchanged, now saving its probabilities; (2) blend. Outputs in `MyDrive/Senior/results/`:
`xgb_e02b.json`, `xgb_e02b_preds.csv.gz`, `blend_e02b.json`. Test block stays sealed.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

Mounted at /content/drive


In [2]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os, xgboost
print('xgboost', xgboost.__version__)
for f in ['scripts/baselines_tabular.py', 'scripts/blend_preds.py']:
    assert os.path.exists(f), f'stop: push {f} from your PC, then run again'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
assert 'predict_proba' in open('scripts/baselines_tabular.py', encoding='utf-8').read(), 'stop: push the latest baselines_tabular.py (saves probabilities)'
assert os.path.exists(f'{OUT_DIR}/main_model_v1_news_p41_preds.csv.gz'), 'stop: E01 predictions not found in Drive results'


/content/Senior-Project
85a55d3 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
xgboost 3.4.1
scripts/baselines_tabular.py e94b996c488d
scripts/blend_preds.py ae587320eb83


In [3]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

OK  data/raw/prices/siamchart/set50_union_2019_2025.csv
OK  data/raw/prices/yahoo/yahoo_events_2019_2025.csv
OK  data/derived/sentiment_scores.csv


In [4]:
!python tests/test_pipeline.py

PASS dividend adjustment removes the XD drop (raw -0.0216 -> adj 0.0011)
PASS split days give normal returns after adjustment (max |r| 0.172)
PASS no return across the TRUE changeover
PASS no return across the SCB changeover
PASS no return across the GULF changeover
PASS no return across the TIDLOR changeover
PASS unexplained GULF 2020-04-16 jump is a break
PASS point-in-time membership: 40-50 members with prices on >95% of days
PASS indicators at t ignore all data after t
PASS fwd_5 at t = sum of returns t+1..t+5
PASS forward returns stop at a changeover
PASS classify has no default sigma or k
PASS fit_k balances classes (k=0.432, {'down': 0.3381, 'flat': 0.3317, 'up': 0.3301})
PASS window = the 60 rows ending at t (inclusive), nothing after t
PASS window never reaches across a changeover
PASS sector table covers all 70 union tickers
PASS blocks are chronological and disjoint
PASS 10 purged dates between train and val
PASS scaler refuses to fit on non-training rows
all pipeline tests 

In [5]:
# 1) XGBoost, same fixed settings as the M2 baseline, now also writing every probability
!python scripts/baselines_tabular.py --out "$OUT_DIR/xgb_e02b.json"


h=1: k=0.307  trend macro-F1 xgb {'mean': 0.35487, 'std': 0.00154} vs majority 0.1697 / random 0.3333 / continues 0.3525 / reverses 0.3727  |  R2_OS xgb {'mean': -0.02315, 'std': 0.00107}
h=5: k=0.318  trend macro-F1 xgb {'mean': 0.38023, 'std': 0.00307} vs majority 0.1698 / random 0.3333 / continues 0.344 / reverses 0.3641  |  R2_OS xgb {'mean': -0.09562, 'std': 0.00029}
h=10: k=0.324  trend macro-F1 xgb {'mean': 0.36057, 'std': 0.00139} vs majority 0.1736 / random 0.3333 / continues 0.3874 / reverses 0.3334  |  R2_OS xgb {'mean': -0.13158, 'std': 0.00315}
wrote /content/drive/MyDrive/Senior/results/xgb_e02b.json and /content/drive/MyDrive/Senior/results/xgb_e02b_preds.csv.gz


In [6]:
# 2) Blend with E01 (w = 0.5, seed s with seed s); prints each input alone, the blend, and the gap to the pass bar
!python scripts/blend_preds.py --deep "$OUT_DIR/main_model_v1_news_p41_preds.csv.gz" --deep-model multiscale_sector \
    --xgb "$OUT_DIR/xgb_e02b_preds.csv.gz" --weight 0.5 --out "$OUT_DIR/blend_e02b.json"


h=1: deep 0.3267  xgb 0.3549  blend 0.3502 +- 0.0047  (bar 0.393, gap -0.0428)  separation {'down': 0.6, 'flat': 3.9, 'up': 2.8}
h=5: deep 0.3439  xgb 0.3802  blend 0.3749 +- 0.0046  (bar 0.4, gap -0.0251)  separation {'down': 1.7, 'flat': 7.0, 'up': 4.0}
h=10: deep 0.3415  xgb 0.3606  blend 0.3576 +- 0.0038  (bar 0.407, gap -0.0494)  separation {'down': 0.2, 'flat': 6.1, 'up': 1.4}
wrote /content/drive/MyDrive/Senior/results/blend_e02b.json
